In [ ]:
!pip install --quiet accelerate
!pip install --quiet bitsandbytes
!pip install --quiet peft
!pip install --quiet trl
!pip install --quiet datasets
!pip install --quiet multiprocess
!pip install --quiet -U nnsight
!pip install --quiet llm-fleet

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from huggingface_hub import hf_hub_download, login

import torch
import numpy as np

import math
import json

from IPython.display import HTML

In [ ]:
device_count = torch.cuda.device_count()
devices = []

for i in range(device_count):
    devices.append(f"cuda:{i}")

In [ ]:
hf_token = '' # use your hf token

In [ ]:
login(token=hf_token)

In [ ]:
repo_id = "microsoft/lost_in_conversation"
filename = "lost_in_conversation.json"

tasks_dataset = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")

In [ ]:
with open(tasks_dataset) as f:
    data = json.load(f)

In [ ]:
coding_tasks = [item for item in data if item.get('task', "") == "code"]
print(coding_tasks[0])

In [ ]:
MODEL_NAME = "google/gemma-3-4b-it"

workers = []
dtype = torch.float16

for i in range(1):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float32
    )
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map={'': f'cuda:{i}'})
    workers.append((model, tokenizer))

tokenizer = workers[0][1]

In [ ]:
generation_kwargs = dict(
    max_new_tokens=1024,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

generation_config = GenerationConfig(
    max_new_tokens=1024,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
prompt_content = """
You are an expert Python programmer. You will be given a question (problem specification) and will generate a correct Python program that matches the specification and passes all tests.

Format:
- [Standalone] Make sure that your answer consists of only one Python function at the top level. Do not wrap with a class or split into multiple functions.
"""

system_prompt = {"role": "system", "content": prompt_content}

def prepare_messages(messages):
  return [system_prompt] + messages

In [ ]:
def format_token(token):
    return token.replace(' ', '&nbsp;').replace('\t', '&nbsp;' * 4)

def visualize_in_html(tokens, values, color=None, key=None):
    html = ""
    for token, v in zip(tokens, values):
        rgba = [0] * 4
        if color is not None:
            for i, c in enumerate(color[:4]):
                rgba[i] = c

        if key is None:
            value = v
        else:
            value = key(v)

        if isinstance(value, (float, int)):
            rgba[3]=value
        else:
            for i, c in enumerate(value[:4]):
                rgba[i] = c


        background_color = f"rgba({rgba[0]}, {rgba[1]}, {rgba[2]}, {rgba[3]:.3f})"

        if token.startswith('\n') and token != '\n':
            html += "<br>"

        html += f"<span style='background-color:{background_color}; padding:2px;' title='{rgba[3]:.3f}'>{format_token(token)}</span>"

        if token.endswith('\n'):
            html += "<br>"

    return html

In [ ]:
model.config

In [ ]:
logit_dim = model.config.text_config.vocab_size

In [ ]:
def remove_all_module_hooks(model: torch.nn.Module) -> None:
    """Clears internal hook dictionaries for all modules."""
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()

In [ ]:
remove_all_module_hooks(model.model.language_model)

In [ ]:
def estimate_decisions(messages, l=-1):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', tokenize=True, add_generation_prompt=True)

    entropies = []
    varentropies = []
    lens_tokens = []
    tokens = []

    def get_layer_n_hook(model, entropies, varentropies, lens_tokens):
        def hook_fn(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            last_token_hidden_state = hidden_states[:, -1:, :].clone()
            with torch.no_grad():
                normed_state = model.model.language_model.norm(last_token_hidden_state)
                logits = model.lm_head(normed_state)
                probs = torch.nn.functional.softmax(logits, dim=-1)
                log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
                entropy = -torch.sum(torch.nan_to_num(probs * log_probs, nan=0.0), dim=-1).cpu()
                varentropy = torch.sum(probs.cpu() * (log_probs.cpu() + entropy.unsqueeze(-1))**2, dim=-1)

            entropies.extend(entropy.flatten().tolist())
            varentropies.extend(varentropy.flatten().tolist())
            lens_tokens.append(logits[0].argmax(dim=-1).item())

            return None
        return hook_fn

    def get_embedding_interception_hook(tokens):
        def embedding_interception_hook(module, args):
            input_ids = args[0]
            input_ids = input_ids.flatten().tolist()

            if len(input_ids) == 1:
                tokens.extend(input_ids)

            return None
        return embedding_interception_hook

    intercept_layer = model.model.language_model.layers[l]
    hs_hook_handle = intercept_layer.register_forward_hook(get_layer_n_hook(model, entropies, varentropies, lens_tokens))
    token_hook_handle = model.model.language_model.embed_tokens.register_forward_pre_hook(get_embedding_interception_hook(tokens))

    input_ids=inputs["input_ids"].to(model.device)
    attention_mask=inputs["attention_mask"].to(model.device)
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        do_sample=False,
        max_new_tokens=1024
    )
    hs_hook_handle.remove()
    token_hook_handle.remove()

    tokens_one_by_one = [tokenizer.decode([t]) for t in tokens]
    tokens_match = [1 if l_t == t else 0 for t, l_t in zip(tokens, lens_tokens)]

    def coloring_closure(data):
        entropy, matches = data
        return 255 * (1 - matches), 0, 255 * matches, (1 - (entropy / np.log(logit_dim)))**2

    scores = [math.sqrt(v * e) for v, e in zip(entropies, varentropies)]
    html = visualize_in_html(tokens_one_by_one, list(zip(scores, tokens_match)), (255, 0, 0), lambda et: coloring_closure(et))
    display(HTML(html))

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING']='1'

In [ ]:
for l in range(len(model.model.language_model.layers)):
    print(l)
    estimate_decisions("What are the 10 most populated countries?", l=l)

Here color shows whether selected layers token matches final layer token (blue) or not (red) and intensity shows whether it was certain about it. Balance between lots of blue and few, but not too few uncertainties (depending on the task) is considered the best layer to pick. Here layer -3 is selected.

In [ ]:
def build_activations_dataset(prompts_dataset, l=-1):
    def get_layer_n_hook(model, activations):
        def hook_fn(module, input, output):
            hidden_states = output[0] if isinstance(output, tuple) else output
            hidden_state = hidden_states[:, -1:, :].clone()
            with torch.no_grad():
                normed_state = model.model.language_model.norm(hidden_state)
                logits = model.lm_head(normed_state)

            activations.append(logits.squeeze().detach().cpu())
            return None
        return hook_fn

    def gather_activations(messages, l):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]

        inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', tokenize=True, add_generation_prompt=True)
        activations = []

        intercept_layer = model.model.language_model.layers[l]
        hs_hook_handle = intercept_layer.register_forward_hook(get_layer_n_hook(model, activations))

        input_ids=inputs["input_ids"].to(model.device)
        attention_mask=inputs["attention_mask"].to(model.device)
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            max_new_tokens=1024
        )
        hs_hook_handle.remove()

        return activations

    all_activations = []
    for prompt in prompts_dataset:
        activations = gather_activations(prompt, l)
        all_activations.extend(activations)

    return all_activations

In [ ]:
prompts = []
for task in coding_tasks[:20]:
    prompt = task.get('question', None)
    if prompt is not None:
        prompts.append(prompt)
        continue

    prompt = task.get('prompt', None)
    if prompt is not None:
        prompts.append(prompt)
        continue

In [ ]:
activations_dataset = build_activations_dataset(prompts, l=-3)
logits_dataset = build_activations_dataset(prompts, l=-1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

def clusterize_activations(activations_dataset):
    entropies = []
    varentropies = []

    for logits in activations_dataset:
        probs = torch.nn.functional.softmax(logits, dim=-1)
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        entropy = -torch.sum(probs * log_probs, dim=-1)
        entropies.append(entropy.item() / np.log(logit_dim))

        varentropy = torch.sum(probs * (log_probs + entropy.unsqueeze(-1))**2, dim=-1)
        varentropies.append(4 * varentropy.item() / np.log(logit_dim) ** 2)

    def plot(X, labels):
        _, ax = plt.subplots(figsize=(10, 4))
        unique_labels = set(labels)
        colors = [plt.cm.Spectral(each) for each in np.linspace(0, 1, len(unique_labels))]

        for k, col in zip(unique_labels, colors):
            if k == -1:
                col = [0, 0, 0, 1]

            class_index = (labels == k).nonzero()[0]
            for ci in class_index:
                ax.plot(
                    X[ci, 0],
                    X[ci, 1] if X.shape[1] > 1 else 0,
                    "x" if k == -1 else "o",
                    markerfacecolor=tuple(col),
                    markeredgecolor="k",
                    markersize=8 # if k == -1 else 1 + 5 * proba_map[ci],
                )

        plt.tight_layout()

    entropies = np.array(entropies)[:, np.newaxis]
    varentropies = np.array(varentropies)[:, np.newaxis]

    hdb = KMeans(n_clusters=2)
    labels = hdb.fit_predict(entropies)
    plot(entropies, labels)
    plt.show()

    hdb = KMeans(n_clusters=2)
    labels = hdb.fit_predict(varentropies)
    plot(varentropies, labels)
    plt.show()

    # Clusterize them as 2D distribution
    X = np.concatenate((entropies, varentropies), axis=1)

    hdb = KMeans(n_clusters=4)
    labels = hdb.fit_predict(X)

    plot(X, labels)
    plt.show()

In [ ]:
clusterize_activations(activations_dataset)

Based on this we select 0.1 threshold for entropy and 0.04 threshold for varentropy.

In [ ]:
def get_threshold_index(activations_dataset, threshold):
  index = []

  for i, logits in enumerate(activations_dataset):
      probs = torch.nn.functional.softmax(logits, dim=-1)
      log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
      entropy = -torch.sum(probs * log_probs, dim=-1) / np.log(logit_dim)
      varentropy = 4 * torch.sum(probs.cpu() * (log_probs + entropy.unsqueeze(-1))**2, dim=-1) / np.log(logit_dim)**2

      if entropy > threshold[0] and varentropy > threshold[1]:
          index.append(i)

  return index

In [ ]:
import pandas as pd

def tune_temperature(activations_dataset, n=10):
    temperatures = np.arange(0.0, 15.0 + 1e-6, 0.1)
    entropies_by_temperature = []

    for t in temperatures:
        for i, logits in enumerate(activations_dataset):
            probs = torch.nn.functional.softmax(logits / t, dim=-1)
            log_probs = torch.nn.functional.log_softmax(logits / t, dim=-1)
            entropy = -torch.sum(probs * log_probs, dim=-1).item() / np.log(logit_dim)

            # bruteforce-capability should be measured as well:
            # what is a probability of top-l
            top_1_probability = torch.max(probs).item()
            # What is a probability of top-n
            top_n, _ = torch.topk(probs, k=n)
            top_n_probability = torch.sum(top_n).item()
            # What is a residual/exploration
            exploration_probability = top_n_probability - top_1_probability

            entropies_by_temperature.append({
                'temperature': t,
                'id': i,
                'value': entropy,
                'metric': 'entropy'
            })

            entropies_by_temperature.append({
                'temperature': t,
                'id': i,
                'value': top_1_probability,
                'metric': 'top_1_probability'
            })

            entropies_by_temperature.append({
                'temperature': t,
                'id': i,
                'value': top_n_probability,
                'metric': f'top_{n}_probability'
            })

            entropies_by_temperature.append({
                'temperature': t,
                'id': i,
                'value': exploration_probability,
                'metric': 'exploration_probability'
            })

    df = pd.DataFrame(entropies_by_temperature)
    sns.lineplot(df, x='temperature', y='value', hue='metric')

    pivot_df = df.pivot_table(index='temperature', columns='metric', values='value', aggfunc='mean')
    pivot_df = pivot_df.sort_index()
    exploration_breakpoint = pivot_df[pivot_df['exploration_probability'] > pivot_df['top_1_probability']].index.min()
    plt.axvline(x=exploration_breakpoint, color='red', linestyle='--', label='Test')
    plt.legend()
    plt.show()

In [ ]:
threshold_index = get_threshold_index(activations_dataset, threshold=(0.1, 0.04))
tune_temperature(torch.stack(logits_dataset)[threshold_index], n=32)

Well, it looks like Gemma is pretty confident in its picks. It is usually advised to pick the temperature where exploration is a top pick by mean value, but here it may be unnecessary as token actions are halved on the first iterations, so we can pick lower temperature.